# 01 - Exploratory Data Analysis: who churns, and why?

**Business question:** which customers are most likely to leave, so retention offers can be targeted?

**Data:** Telco Customer Churn, 7,043 customers, 20 features. Overall churn is **26.5%**.

In [1]:
import sys; sys.path.append('../src')
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from data import load_clean
from eda import churn_by
sns.set_theme(style='whitegrid')
df = load_clean()
print(df.shape, f"churn rate: {df['Churn'].mean():.1%}")
df.head()

(7043, 20) churn rate: 26.5%


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## 1. Contract type is the strongest single signal

In [2]:
churn_by(df, 'Contract')

,customers,churn_rate
Contract,,
Month-to-month,3875,0.427097
One year,1473,0.112695
Two year,1695,0.028319


**So what:** month-to-month customers churn at **42.7%**, versus 11.3% on one-year and 2.8% on two-year contracts. Moving month-to-month customers onto longer contracts is the most direct retention lever.

## 2. Churn is highest early in the customer lifetime

In [3]:
df['TenureBand'] = pd.cut(df['tenure'], [-1, 6, 12, 24, 48, 72])
rate = df.groupby('TenureBand', observed=True)['Churn'].mean()
rate.plot(kind='bar', color='#3b6ea5'); plt.ylabel('Churn rate'); plt.show()
rate.round(3)

C:\Users\Rohit Maheshwari\AppData\Local\Temp\ipykernel_2896\2712616248.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  rate.plot(kind='bar', color='#3b6ea5'); plt.ylabel('Churn rate'); plt.show()


TenureBand
(-1, 6]     0.529
(6, 12]     0.359
(12, 24]    0.287
(24, 48]    0.204
(48, 72]    0.095
Name: Churn, dtype: float64

**So what:** customers in their first 6 months churn at **52.9%**; after 4 years it falls to **9.5%**. Onboarding and the first-year experience deserve the retention budget.

## 3. Service and payment drivers

In [4]:
for c in ['InternetService', 'PaymentMethod', 'TechSupport', 'OnlineSecurity']:
    display(churn_by(df, c))

,customers,churn_rate
InternetService,,
Fiber optic,3096,0.418928
DSL,2421,0.189591
No,1526,0.074050


,customers,churn_rate
PaymentMethod,,
Electronic check,2365,0.452854
Mailed check,1612,0.191067
Bank transfer (automatic),1544,0.167098
Credit card (automatic),1522,0.152431


,customers,churn_rate
TechSupport,,
No,3473,0.416355
Yes,2044,0.151663
No internet service,1526,0.074050


,customers,churn_rate
OnlineSecurity,,
No,3498,0.417667
Yes,2019,0.146112
No internet service,1526,0.074050


**So what:**
- **Fiber optic** customers churn at **41.9%** (DSL 19.0%). Worth checking price/quality of fiber.
- **Electronic check** payers churn at **45.3%**; automatic payment methods sit at 15-17%. Auto-pay may reflect commitment, so nudging customers to auto-pay is a cheap test.
- Customers **without TechSupport / OnlineSecurity** churn at ~42% versus ~15% with them.

## 4. Price and demographics

In [5]:
print(df.groupby('Churn')['MonthlyCharges'].mean().round(2))
display(churn_by(df, 'SeniorCitizen'))
display(churn_by(df, 'Partner'))
display(churn_by(df, 'gender'))

Churn
0    61.27
1    74.44
Name: MonthlyCharges, dtype: float64


,customers,churn_rate
SeniorCitizen,,
1,1142,0.416813
0,5901,0.236062


,customers,churn_rate
Partner,,
No,3641,0.329580
Yes,3402,0.196649


,customers,churn_rate
gender,,
Female,3488,0.269209
Male,3555,0.261603


**So what:** churners pay more per month (**$74.44** vs $61.27). Senior citizens (41.7%) and customers without a partner (33.0%) churn more. **Gender makes no difference** (26.9% vs 26.2%), so it is not a useful feature and should not drive decisions.

## Takeaways for modeling
1. The highest-risk profile is: month-to-month, short tenure, fiber optic, electronic check, no tech support.
2. Classes are imbalanced (26.5% churn), so use AUC/recall and a cost-based threshold, not accuracy.
3. `TotalCharges` is largely `tenure x MonthlyCharges`; watch for collinearity.
4. Split the data **before** any fitting to avoid leakage.